In [1]:
import ssl
import certifi
from urllib.request import urlopen
from bs4 import BeautifulSoup
import csv
import unicodedata
import pandas as pd

In [2]:
start_url = "https://mailarchive.upatras.gr/search/20231231.151927.4ea97190@.en.html"

In [3]:
def create_new_excel():
    fields=['Author','Date','To','Subject','Message']
    filename='test_dataset.csv'
    with open(filename,'w',encoding='utf-32', newline='') as csvfile:
        csvwriter=csv.writer(csvfile,delimiter='\t')
        csvwriter.writerow(fields)

In [4]:
def scrapper(url):
    page = urlopen(url, context=ssl.create_default_context(cafile=certifi.where()))
    soup =  BeautifulSoup(page, "html.parser" )
    links=soup.find_all('a')
    urls=[]
    for link in links[3:-3]:
        a=link['href']
        new_url="https://mailarchive.upatras.gr"+a[2:None]
        urls.append(new_url)
    for urli in urls:
        try:
            new_page = urlopen(urli, context=ssl.create_default_context(cafile=certifi.where()))
            new_soup =  BeautifulSoup(new_page, "html.parser" )
            raw_text=new_soup.find('div',{"class": "messageBody"})
            text=raw_text.get_text()
            raw_info=new_soup.find('div',{"class": None})
            info=raw_info.get_text()
            norm_info = unicodedata.normalize('NFKD', info).encode('utf-32', 'ignore').decode('utf-32')
            x=norm_info.split('Author:')
            y=x[-1].split('Date:')
            z=y[-1].split('To:')
            w=z[-1].split('Subject:')
            data=[y[0],z[0],w[0],w[-1],text]
            with open('test_dataset.csv','a',encoding='utf-32', newline='') as csvfile:
                writer = csv.writer(csvfile,delimiter='\t')
                writer.writerow(data)
        except:
            continue

In [5]:
def find_next_url(url):
    page = urlopen(url, context=ssl.create_default_context(cafile=certifi.where()))
    soup =  BeautifulSoup(page, "html.parser" )
    kl = soup.find_all('td', {'align': 'right'})
    next_url=kl[1].find('a')['href']
    next_url='https://mailarchive.upatras.gr/search/'+next_url
    return next_url

In [6]:
create_new_excel()

In [7]:
url=start_url
for i in range(1000):
    scrapper(url)
    url=find_next_url(url)

TypeError: 'NoneType' object is not subscriptable